##### Copyright 2023 The MediaPipe Authors. All Rights Reserved.
##### Modifications 2026 by: Eli Guo, Tim Zhou, Iris Wu (NYU Center for Data Science)

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multi-Modal Human Landmark Detection with MediaPipe Tasks

This notebook runs pose, face, and hand landmark detection on videos using
the MediaPipe Tasks API. It draws all landmarks on the same output video and
optionally exports per-frame landmarks to JSON for further analysis.

## Preparation

Install MediaPipe and download the task model bundles for pose, face, and hand landmarks.  
Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/overview) for more information about these models and their options.

In [2]:
!pip install -q mediapipe

# pose
!wget -O pose_landmarker.task -q \
  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

# face
!wget -O face_landmarker_v2_with_blendshapes.task -q \
  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

# hand
!wget -O hand_landmarker.task -q \
  https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [3]:
import json
from typing import Any, Dict, List, Optional

import cv2
import numpy as np
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

In [4]:
import os
print("hostname:", os.uname().nodename)
!nvidia-smi

hostname: a100-4004
Sun Apr  5 22:47:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  NVIDIA A100 80GB PCIe          On  |   00000000:17:00.0 Off |                    0 |
| N/A   31C    P0             51W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+------------------------+----------------------+
                                                                                         
+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                     

## Visualization utilities

Define helper functions to convert MediaPipe landmark structures into simple Python dictionaries and to draw pose, face, and hand landmarks on RGB images using the MediaPipe Tasks drawing utilities.

These functions will be used later to visualize the detection results frame by frame.

In [5]:
def landmark_to_dict(lm) -> Dict[str, float]:
    d: Dict[str, float] = {
        "x": float(lm.x),
        "y": float(lm.y),
        "z": float(lm.z),
    }

    vis = getattr(lm, "visibility", None)
    if vis is not None:
        d["visibility"] = float(vis)

    pres = getattr(lm, "presence", None)
    if pres is not None:
        d["presence"] = float(pres)

    return d


def landmark_list_to_dicts(landmarks) -> List[Dict[str, float]]:
    return [landmark_to_dict(lm) for lm in landmarks]

In [6]:
def draw_pose_landmarks_on_image(
    rgb_image: np.ndarray,
    detection_result: vision.PoseLandmarkerResult,
) -> np.ndarray:
    """Draw pose landmarks on an RGB image using MediaPipe Tasks drawing utils."""
    pose_landmarks_list = detection_result.pose_landmarks
    annotated = np.copy(rgb_image)

    if not pose_landmarks_list:
        return annotated

    # default landmark style from Tasks API
    pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
    # simple connection style (you used this pattern in the original pose notebook)
    pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

    for pose_landmarks in pose_landmarks_list:
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=pose_landmark_style,
            connection_drawing_spec=pose_connection_style,
        )

    return annotated

In [ ]:
def draw_face_landmarks_on_image(
    rgb_image: np.ndarray,
    detection_result: vision.FaceLandmarkerResult,
) -> np.ndarray:
    """draw face mesh landmarks on an rgb image."""
    face_landmarks_list = detection_result.face_landmarks
    annotated = np.copy(rgb_image)

    if not face_landmarks_list:
        return annotated

    for face_landmarks in face_landmarks_list:
        # tessellation
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_tesselation_style(),
        )
        # contours
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style(),
        )

    return annotated

In [8]:
MARGIN = 10          # pixels
FONT_SIZE = 1.0
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green


def draw_hand_landmarks_on_image(
    rgb_image: np.ndarray,
    detection_result: vision.HandLandmarkerResult,
) -> np.ndarray:
    """draw hand landmarks and handedness on an rgb image."""
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated = np.copy(rgb_image)

    if not hand_landmarks_list:
        return annotated

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        handedness = handedness_list[idx] if handedness_list else None

        # draw hand skeleton
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=hand_landmarks,
            connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
            landmark_drawing_spec=drawing_styles.get_default_hand_landmarks_style(),
            connection_drawing_spec=drawing_styles.get_default_hand_connections_style(),
        )

        # draw handedness label (left / right)
        h, w, _ = annotated.shape
        xs = [lm.x for lm in hand_landmarks]
        ys = [lm.y for lm in hand_landmarks]
        text_x = int(min(xs) * w)
        text_y = int(min(ys) * h) - MARGIN

        label = handedness[0].category_name if handedness else ""
        cv2.putText(
            annotated,
            label,
            (text_x, max(text_y, MARGIN)),
            cv2.FONT_HERSHEY_DUPLEX,
            FONT_SIZE,
            HANDEDNESS_TEXT_COLOR,
            FONT_THICKNESS,
            cv2.LINE_AA,
        )

    return annotated

## Creating the tasks

Define small factory functions that create the Pose Landmarker, Face Landmarker, and Hand Landmarker in VIDEO running mode.

Each function configures the corresponding task with a model path, the maximum number of instances, and the relevant detection, presence, and tracking confidence thresholds.

In [9]:
def create_pose_landmarker(
    model_path: str,
    num_poses: int,
    min_pose_detection_confidence: float,
    min_pose_presence_confidence: float,
    min_pose_tracking_confidence: float,
) -> vision.PoseLandmarker:
    """Create a PoseLandmarker in VIDEO mode with the given thresholds."""
    BaseOptions = mp.tasks.BaseOptions
    RunningMode = mp.tasks.vision.RunningMode

    base_options = BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=RunningMode.VIDEO,
        num_poses=num_poses,
        min_pose_detection_confidence=min_pose_detection_confidence,
        min_pose_presence_confidence=min_pose_presence_confidence,
        min_tracking_confidence=min_pose_tracking_confidence,
        output_segmentation_masks=False,
    )
    return vision.PoseLandmarker.create_from_options(options)


def create_face_landmarker(
    model_path: str,
    num_faces: int,
    min_face_detection_confidence: float,
    min_face_presence_confidence: float,
    min_face_tracking_confidence: float,
) -> vision.FaceLandmarker:
    """Create a FaceLandmarker in VIDEO mode with the given thresholds."""
    BaseOptions = mp.tasks.BaseOptions
    RunningMode = mp.tasks.vision.RunningMode

    base_options = BaseOptions(model_asset_path=model_path)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=RunningMode.VIDEO,
        num_faces=num_faces,
        min_face_detection_confidence=min_face_detection_confidence,
        min_face_presence_confidence=min_face_presence_confidence,
        min_tracking_confidence=min_face_tracking_confidence,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    return vision.FaceLandmarker.create_from_options(options)


def create_hand_landmarker(
    model_path: str,
    num_hands: int,
    min_hand_detection_confidence: float,
    min_hand_presence_confidence: float,
    min_hand_tracking_confidence: float,
) -> vision.HandLandmarker:
    """Create a HandLandmarker in VIDEO mode with the given thresholds."""
    BaseOptions = mp.tasks.BaseOptions
    RunningMode = mp.tasks.vision.RunningMode

    base_options = BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        running_mode=RunningMode.VIDEO,
        num_hands=num_hands,
        min_hand_detection_confidence=min_hand_detection_confidence,
        min_hand_presence_confidence=min_hand_presence_confidence,
        min_tracking_confidence=min_hand_tracking_confidence,
    )
    return vision.HandLandmarker.create_from_options(options)

## Representing per-frame results

Define a function that converts the pose, face, and hand Landmarker results for a single frame into a JSON-compatible dictionary.

The record stores the frame index, timestamp, image-space landmarks, optional world-space landmarks, face blendshapes, and hand handedness.

In [ ]:
def frame_results_to_record(
    frame_index: int,
    timestamp_ms: int,
    pose_result: vision.PoseLandmarkerResult,
    face_result: vision.FaceLandmarkerResult,
    hand_result: vision.HandLandmarkerResult,
) -> Dict[str, Any]:
    frame_data: Dict[str, Any] = {
        "frame_index": frame_index,
        "timestamp_ms": timestamp_ms,
    }

    # poses
    poses_list: List[Dict[str, Any]] = []
    for i, pose_lms in enumerate(pose_result.pose_landmarks):
        pose_entry: Dict[str, Any] = {
            "landmarks": landmark_list_to_dicts(pose_lms),
        }
        if i < len(pose_result.pose_world_landmarks):
            pose_entry["world_landmarks"] = landmark_list_to_dicts(
                pose_result.pose_world_landmarks[i]
            )
        poses_list.append(pose_entry)
    frame_data["poses"] = poses_list

    # faces
    faces_list: List[Dict[str, Any]] = []
    for i, face_lms in enumerate(face_result.face_landmarks):
        face_entry: Dict[str, Any] = {
            "landmarks": landmark_list_to_dicts(face_lms),
        }
        if face_result.face_blendshapes and i < len(face_result.face_blendshapes):
            bs = face_result.face_blendshapes[i]
            face_entry["blendshapes"] = [
                {
                    "category_name": c.category_name,
                    "score": float(c.score),
                    "index": c.index,
                }
                for c in bs
            ]
        faces_list.append(face_entry)
    frame_data["faces"] = faces_list

    # hands
    hands_list: List[Dict[str, Any]] = []
    for i, hand_lms in enumerate(hand_result.hand_landmarks):
        hand_entry: Dict[str, Any] = {
            "landmarks": landmark_list_to_dicts(hand_lms),
        }
        if hand_result.hand_world_landmarks and i < len(hand_result.hand_world_landmarks):
            hand_entry["world_landmarks"] = landmark_list_to_dicts(
                hand_result.hand_world_landmarks[i]
            )
        if hand_result.handedness and i < len(hand_result.handedness):
            handedness = hand_result.handedness[i]
            hand_entry["handedness"] = [
                {
                    "index": h.index,
                    "score": float(h.score),
                    "category_name": h.category_name,
                    "display_name": h.display_name,
                }
                for h in handedness
            ]
        hands_list.append(hand_entry)
    frame_data["hands"] = hands_list

    return frame_data

## Holistic video processing

Implement the main `run_holistic_on_video` function.

This function opens a video file, runs pose, face, and hand landmark detection on each frame in VIDEO mode, draws all detected landmarks on the frame, and writes the annotated frames to an output video. Optionally, it aggregates the per-frame records into a single JSON structure and saves it to disk.

In [ ]:
def run_holistic_on_video(
    input_path: str,
    output_video_path: str,
    json_output_path: Optional[str] = None,
    pose_model_path: str = "pose_landmarker.task",
    face_model_path: str = "face_landmarker_v2_with_blendshapes.task",
    hand_model_path: str = "hand_landmarker.task",
    num_poses: int = 2,
    num_faces: int = 2,
    num_hands: int = 2,
    # pose thresholds
    min_pose_detection_confidence: float = 0.5,
    min_pose_presence_confidence: float = 0.5,
    min_pose_tracking_confidence: float = 0.5,
    # face thresholds
    min_face_detection_confidence: float = 0.5,
    min_face_presence_confidence: float = 0.5,
    min_face_tracking_confidence: float = 0.5,
    # hand thresholds
    min_hand_detection_confidence: float = 0.5,
    min_hand_presence_confidence: float = 0.5,
    min_hand_tracking_confidence: float = 0.5,
) -> None:
    """Run pose + face + hand landmarkers on a video and save:
       1) an annotated video
       2) (optionally) a JSON file with all landmarks per frame.
    """

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"[ERROR] could not open video: {input_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"[INFO] input video: {input_path}")
    print(f"[INFO] resolution: {width}x{height}, fps: {fps}, frames: {frame_count}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    if not out.isOpened():
        print(f"[ERROR] could not open video writer for: {output_video_path}")
        cap.release()
        return

    frames_json: List[Dict[str, Any]] = []
    mp_image_format = mp.ImageFormat.SRGB

    # create three detectors (Tasks API only)
    with create_pose_landmarker(
        pose_model_path,
        num_poses,
        min_pose_detection_confidence,
        min_pose_presence_confidence,
        min_pose_tracking_confidence,
    ) as pose_detector, create_face_landmarker(
        face_model_path,
        num_faces,
        min_face_detection_confidence,
        min_face_presence_confidence,
        min_face_tracking_confidence,
    ) as face_detector, create_hand_landmarker(
        hand_model_path,
        num_hands,
        min_hand_detection_confidence,
        min_hand_presence_confidence,
        min_hand_tracking_confidence,
    ) as hand_detector:

        frame_index = 0

        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp_image_format, data=frame_rgb)

            timestamp_ms = int(1000 * frame_index / fps)

            # run all three models on this frame
            pose_result = pose_detector.detect_for_video(mp_image, timestamp_ms)
            face_result = face_detector.detect_for_video(mp_image, timestamp_ms)
            hand_result = hand_detector.detect_for_video(mp_image, timestamp_ms)

            # draw overlays
            annotated_rgb = frame_rgb
            annotated_rgb = draw_pose_landmarks_on_image(annotated_rgb, pose_result)
            annotated_rgb = draw_face_landmarks_on_image(annotated_rgb, face_result)
            annotated_rgb = draw_hand_landmarks_on_image(annotated_rgb, hand_result)

            annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
            out.write(annotated_bgr)

            # collect json for this frame
            frame_record = frame_results_to_record(
                frame_index, timestamp_ms, pose_result, face_result, hand_result
            )
            frames_json.append(frame_record)

            frame_index += 1
            if frame_index % 50 == 0:
                print(f"[INFO] processed {frame_index}/{frame_count} frames")

    cap.release()
    out.release()
    print(f"[INFO] done. saved annotated video to: {output_video_path}")

    if json_output_path is not None:
        video_json = {
            "input_video": input_path,
            "fps": fps,
            "width": width,
            "height": height,
            "frame_count": frame_count,
            "frames": frames_json,
        }
        with open(json_output_path, "w", encoding="utf-8") as f:
            json.dump(video_json, f)
        print(f"[INFO] saved landmarks json to: {json_output_path}")

## Running the holistic pipeline on sample videos

Apply the holistic pipeline to two sample videos.

For each input video, the notebook produces an annotated MP4 file and a JSON file containing the complete sequence of pose, face, and hand landmarks for all frames.

In [12]:
run_holistic_on_video(
    input_path="/gpfs/data/shenlab/data/pediatric_psychiatry/COPE_Videos_PHI_confidential/6_Months/M22272/E1/M22272_6_Freeplay.mov",
    output_video_path="outputs/M22272_6_Freeplay.mp4",
    json_output_path="outputs/M22272_6_Freeplay.json",
    num_poses=2,
    num_faces=2,
    num_hands=4,
    min_face_detection_confidence=1e-3,
    min_face_presence_confidence=0.1,
    min_face_tracking_confidence=0.1,
    min_hand_detection_confidence=0.3,
    min_hand_tracking_confidence=0.3
)

run_holistic_on_video(
    input_path="/gpfs/data/shenlab/data/pediatric_psychiatry/COPE_Videos_PHI_confidential/6_Months/M22272/E1/M22272_6_Nail_Clipping.mov",
    output_video_path="outputs/M22272_6_Nail_Clipping.mp4",
    json_output_path="outputs/M22272_6_Nail_Clipping.json",
    num_poses=2,
    num_faces=2,
    num_hands=4,
    min_face_detection_confidence=1e-3,
    min_face_presence_confidence=0.1,
    min_face_tracking_confidence=0.1,
    min_hand_detection_confidence=0.3,
    min_hand_tracking_confidence=0.3
)

run_holistic_on_video(
    input_path="/gpfs/data/shenlab/data/pediatric_psychiatry/COPE_Videos_PHI_confidential/6_Months/M22272/E1/M22272_6_Still_Face.mov",
    output_video_path="outputs/M22272_6_Still_Face.mp4",
    json_output_path="outputs/M22272_6_Still_Face.json",
    num_poses=2,
    num_faces=2,
    num_hands=4,
    min_face_detection_confidence=1e-3,
    min_face_presence_confidence=0.1,
    min_face_tracking_confidence=0.1,
    min_hand_detection_confidence=0.3,
    min_hand_tracking_confidence=0.3
)

run_holistic_on_video(
    input_path="/gpfs/data/shenlab/data/pediatric_psychiatry/COPE_Videos_PHI_confidential/6_Months/M22272/E1/M22272_6_Visual_Attention.mov",
    output_video_path="outputs/M22272_6_Visual_Attention.mp4",
    json_output_path="outputs/M22272_6_Visual_Attention.json",
    num_poses=2,
    num_faces=2,
    num_hands=4,
    min_face_detection_confidence=1e-3,
    min_face_presence_confidence=0.1,
    min_face_tracking_confidence=0.1,
    min_hand_detection_confidence=0.3,
    min_hand_tracking_confidence=0.3
)

[INFO] input video: /gpfs/data/shenlab/data/pediatric_psychiatry/COPE_Videos_PHI_confidential/6_Months/M22272/E1/M22272_6_Freeplay.mov
[INFO] resolution: 1568x874, fps: 59.99840349103294, frames: 37581


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775443653.643303 4130990 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775443653.722794 4130990 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775443653.724270 4131038 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1775443653.733883 4131043 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775443653.753820 4131043 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775443653.789119 4131088 inference_feedback_manager.cc:114] Feedback mana

[INFO] processed 50/37581 frames
[INFO] processed 100/37581 frames
[INFO] processed 150/37581 frames
[INFO] processed 200/37581 frames
[INFO] processed 250/37581 frames
[INFO] processed 300/37581 frames
[INFO] processed 350/37581 frames
[INFO] processed 400/37581 frames
[INFO] processed 450/37581 frames
[INFO] processed 500/37581 frames
[INFO] processed 550/37581 frames
[INFO] processed 600/37581 frames
[INFO] processed 650/37581 frames
[INFO] processed 700/37581 frames
[INFO] processed 750/37581 frames
[INFO] processed 800/37581 frames
[INFO] processed 850/37581 frames
[INFO] processed 900/37581 frames
[INFO] processed 950/37581 frames
[INFO] processed 1000/37581 frames
[INFO] processed 1050/37581 frames
[INFO] processed 1100/37581 frames
[INFO] processed 1150/37581 frames
[INFO] processed 1200/37581 frames
[INFO] processed 1250/37581 frames
[INFO] processed 1300/37581 frames
[INFO] processed 1350/37581 frames
[INFO] processed 1400/37581 frames
[INFO] processed 1450/37581 frames
[INFO

W0000 00:00:1775448645.830263 4138856 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775448645.895552 4138856 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775448645.897010 4138903 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1775448645.903394 4138904 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775448645.914961 4138904 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775448645.934019 4138954 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. 

[INFO] processed 50/6979 frames
[INFO] processed 100/6979 frames
[INFO] processed 150/6979 frames
[INFO] processed 200/6979 frames
[INFO] processed 250/6979 frames
[INFO] processed 300/6979 frames
[INFO] processed 350/6979 frames
[INFO] processed 400/6979 frames
[INFO] processed 450/6979 frames
[INFO] processed 500/6979 frames
[INFO] processed 550/6979 frames
[INFO] processed 600/6979 frames
[INFO] processed 650/6979 frames
[INFO] processed 700/6979 frames
[INFO] processed 750/6979 frames
[INFO] processed 800/6979 frames
[INFO] processed 850/6979 frames
[INFO] processed 900/6979 frames
[INFO] processed 950/6979 frames
[INFO] processed 1000/6979 frames
[INFO] processed 1050/6979 frames
[INFO] processed 1100/6979 frames
[INFO] processed 1150/6979 frames
[INFO] processed 1200/6979 frames
[INFO] processed 1250/6979 frames
[INFO] processed 1300/6979 frames
[INFO] processed 1350/6979 frames
[INFO] processed 1400/6979 frames
[INFO] processed 1450/6979 frames
[INFO] processed 1500/6979 frames


W0000 00:00:1775449491.433902 4140323 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775449491.497005 4140356 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775449491.497995 4140370 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1775449491.503319 4140371 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775449491.514569 4140371 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775449491.533123 4140421 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. 

[INFO] processed 50/24443 frames
[INFO] processed 100/24443 frames
[INFO] processed 150/24443 frames
[INFO] processed 200/24443 frames
[INFO] processed 250/24443 frames
[INFO] processed 300/24443 frames
[INFO] processed 350/24443 frames
[INFO] processed 400/24443 frames
[INFO] processed 450/24443 frames
[INFO] processed 500/24443 frames
[INFO] processed 550/24443 frames
[INFO] processed 600/24443 frames
[INFO] processed 650/24443 frames
[INFO] processed 700/24443 frames
[INFO] processed 750/24443 frames
[INFO] processed 800/24443 frames
[INFO] processed 850/24443 frames
[INFO] processed 900/24443 frames
[INFO] processed 950/24443 frames
[INFO] processed 1000/24443 frames
[INFO] processed 1050/24443 frames
[INFO] processed 1100/24443 frames
[INFO] processed 1150/24443 frames
[INFO] processed 1200/24443 frames
[INFO] processed 1250/24443 frames
[INFO] processed 1300/24443 frames
[INFO] processed 1350/24443 frames
[INFO] processed 1400/24443 frames
[INFO] processed 1450/24443 frames
[INFO

W0000 00:00:1775453037.452395 4146509 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775453037.517493 4146509 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775453037.518553 4146547 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1775453037.524135 4146548 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775453037.534541 4146564 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775453037.553025 4146621 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. 

[INFO] processed 50/6620 frames
[INFO] processed 100/6620 frames
[INFO] processed 150/6620 frames
[INFO] processed 200/6620 frames
[INFO] processed 250/6620 frames
[INFO] processed 300/6620 frames
[INFO] processed 350/6620 frames
[INFO] processed 400/6620 frames
[INFO] processed 450/6620 frames
[INFO] processed 500/6620 frames
[INFO] processed 550/6620 frames
[INFO] processed 600/6620 frames
[INFO] processed 650/6620 frames
[INFO] processed 700/6620 frames
[INFO] processed 750/6620 frames
[INFO] processed 800/6620 frames
[INFO] processed 850/6620 frames
[INFO] processed 900/6620 frames
[INFO] processed 950/6620 frames
[INFO] processed 1000/6620 frames
[INFO] processed 1050/6620 frames
[INFO] processed 1100/6620 frames
[INFO] processed 1150/6620 frames
[INFO] processed 1200/6620 frames
[INFO] processed 1250/6620 frames
[INFO] processed 1300/6620 frames
[INFO] processed 1350/6620 frames
[INFO] processed 1400/6620 frames
[INFO] processed 1450/6620 frames
[INFO] processed 1500/6620 frames


In [13]:
import json

json_path = "MotorVideo_GetUpGo_0AEBFF6F-F811-4633-8219-C30FDABD4B0D_holistic.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))
print("input_video:", data["input_video"])
print("fps:", data["fps"])
print("resolution:", data["width"], "x", data["height"])
print("frame_count:", data["frame_count"])

frames = data["frames"]
print("Number of frame records in JSON:", len(frames))

Top-level keys: ['input_video', 'fps', 'width', 'height', 'frame_count', 'frames']
input_video: ./data/MotorVideo_GetUpGo_0AEBFF6F-F811-4633-8219-C30FDABD4B0D_.mov
fps: 29.99880100715399
resolution: 1920 x 1080
frame_count: 1251
Number of frame records in JSON: 1251


In [14]:
frame = data["frames"][1099]
print(frame.keys())
print(len(frame["poses"]))
print(len(frame["faces"]))
print(len(frame["hands"]))

print(frame["poses"][0]["landmarks"][0])
print(frame["hands"][0]["landmarks"][0])

dict_keys(['frame_index', 'timestamp_ms', 'poses', 'faces', 'hands'])
1
0
2
{'x': 0.49645906686782837, 'y': 0.1235024631023407, 'z': -0.037421371787786484, 'visibility': 0.999991774559021, 'presence': 0.9999964237213135}
{'x': 0.47063300013542175, 'y': 0.35356616973876953, 'z': 5.127072100208352e-08}


In [15]:
import json

json_path = "Still_face_experiment_holistic.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))
print("input_video:", data["input_video"])
print("fps:", data["fps"])
print("resolution:", data["width"], "x", data["height"])
print("frame_count:", data["frame_count"])

frames = data["frames"]
print("Number of frame records in JSON:", len(frames))

Top-level keys: ['input_video', 'fps', 'width', 'height', 'frame_count', 'frames']
input_video: ./data/Still face experiment - Lise-Lotte Austad (720p, h264).mp4
fps: 29.97002997002997
resolution: 1280 x 720
frame_count: 3191
Number of frame records in JSON: 3188


In [16]:
indices = [f["frame_index"] for f in frames]
print("min index:", min(indices))
print("max index:", max(indices))
print("num indices:", len(indices))
print("are indices contiguous?", max(indices) - min(indices) + 1 == len(indices))

min index: 0
max index: 3187
num indices: 3188
are indices contiguous? True


In [17]:
frame = data["frames"][720]
print(frame.keys())
print(len(frame["poses"]))
print(len(frame["faces"]))
print(len(frame["hands"]))

print(frame["poses"][0]["landmarks"][0])
print(frame["faces"][0]["landmarks"][0])
print(frame["hands"][0]["landmarks"][0])

dict_keys(['frame_index', 'timestamp_ms', 'poses', 'faces', 'hands'])
2
2
1
{'x': 0.2359851896762848, 'y': 0.5677461624145508, 'z': -0.1754406988620758, 'visibility': 0.9999454021453857, 'presence': 0.9999979734420776}
{'x': 0.8767356872558594, 'y': 0.20418982207775116, 'z': -0.010764705948531628}
{'x': 0.7335441708564758, 'y': 0.490418016910553, 'z': 2.2013580291968537e-07}
